# MeshVTON v2 — Faz 1: Zero-shot taban çizgisi\n\nÜç hücre: kurulum / koşu / sonuç. **Mantık notebook'ta değil, `v2/` paketindedir** — değişiklik gerekiyorsa script/modül düzenlenir, hücre değil.

In [ ]:
#@title 1) Kurulum — HIZLI sürüm (~2-3 dk; derleme YOK, pytorch3d GEREKMEZ)
import os
if not os.path.exists('/content/MeshVTON'):
    !git clone https://github.com/SerhanTelatar/MeshVTON /content/MeshVTON
%cd /content/MeshVTON
!git pull

# Colab'da hazır olanları (torch, torchvision, transformers, accelerate, opencv,
# skimage, PIL, yaml...) YENİDEN KURMA — yalnız eksikler:
!pip -q install "diffusers>=0.34" "peft>=0.14" lpips einops sentencepiece trimesh smplx pyrender onnxruntime-gpu

# Faz 1 render'ı pyrender/EGL kullanır (headless GPU); pytorch3d Faz 2'de gelecek.
os.environ['PYOPENGL_PLATFORM'] = 'egl'

# IDM-VTON yalnız KIŞI ön-işlemesi için (parsing onnx + openpose ckpt) — diffusion tarafı kullanılmaz
if not os.path.exists('/content/IDM-VTON'):
    !git clone -q https://github.com/yisol/IDM-VTON /content/IDM-VTON
!mkdir -p /content/IDM-VTON/ckpt/humanparsing /content/IDM-VTON/ckpt/openpose/ckpts
!wget -q -nc -O /content/IDM-VTON/ckpt/humanparsing/parsing_atr.onnx https://huggingface.co/yisol/IDM-VTON/resolve/main/humanparsing/parsing_atr.onnx
!wget -q -nc -O /content/IDM-VTON/ckpt/humanparsing/parsing_lip.onnx https://huggingface.co/yisol/IDM-VTON/resolve/main/humanparsing/parsing_lip.onnx
!wget -q -nc -O /content/IDM-VTON/ckpt/openpose/ckpts/body_pose_model.pth https://huggingface.co/yisol/IDM-VTON/resolve/main/openpose/ckpts/body_pose_model.pth

# FLUX.1 dev modelleri gated: token'ı Colab Secrets'a HF_TOKEN olarak ekleyin (soldaki anahtar simgesi)
# — interaktif login() beklemesi yok. Secrets yoksa aşağıdaki satırı açın:
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# from huggingface_hub import login; login()

from google.colab import drive
drive.mount('/content/drive')
print('OK — veri yollarını kontrol edin: data/garments_3d + VITON-HD test/image')

In [ ]:
#@title 2) Golden set (ilk kez) + zero-shot koşuları
VITONHD_TEST = "/content/MeshVTON/data/vitonhd/test/image"  #@param {type:"string"}
GARMENTS     = "/content/MeshVTON/data/garments_3d"          #@param {type:"string"}
VARIANT      = "fill_spatial"  #@param ["fill_spatial", "kontext"]
LIMIT        = 4  #@param {type:"integer"}  # 0 = tam koşu (100 kombo)

import os
if not os.path.exists('v2/data/golden/manifest.json'):
    !python v2/scripts/build_golden_set.py --vitonhd-test "$VITONHD_TEST" --garments "$GARMENTS"

limit_arg = f"--limit {LIMIT}" if LIMIT else ""
!python v2/scripts/zero_shot_baseline.py --variant $VARIANT --idm-repo /content/IDM-VTON $limit_arg

In [ ]:
#@title 3) Sonuçlar
from IPython.display import Image as I, display, Markdown
import pathlib
for v in ("fill_spatial", "kontext"):
    md = pathlib.Path(f"v2/eval_results/phase1_{v}.md")
    grid = pathlib.Path(f"v2/eval_results/phase1_{v}_grid.png")
    if md.exists():
        display(Markdown(f"## {v}"), Markdown(md.read_text()))
    if grid.exists():
        display(I(str(grid)))